# Notebook 1: EDA & Domain Context
## ESP Predictive Maintenance — Electric Submersible Pumps

**Author**: Your Name | BSc Oil & Gas + MSc Data Science

### Learning objectives
1. Understand the operating context of ESPs in oil and gas wells
2. Explore raw sensor distributions, correlations, and failure patterns
3. Identify domain-relevant signals for each failure mode
4. Visualise the class imbalance problem

### Domain context
Electric Submersible Pumps (ESPs) are multistage centrifugal pumps deployed inside oil wells to provide artificial lift. They operate continuously at 2,000–3,500+ RPM in harsh downhole conditions:
- Temperatures: 100–200°C
- Pressures: 1,000–5,000+ PSI
- Fluid: Oil/water/gas mixture with sand and scale potential

**Key failure modes**:
| Failure | Root cause | Sensor signature |
|---------|-----------|------------------|
| Gas locking | Free gas entering impellers | Current drop, oscillating pressure |
| Abrasive wear | Sand/solids erosion | Gradual vibration increase |
| Motor overheating | Winding insulation degradation | Temperature rise, resistance change |
| Scale buildup | Carbonate/sulfate deposition | Differential pressure increase |


In [ ]:
# ── Cell 1: Install dependencies (Colab) ──────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pandas numpy matplotlib seaborn scikit-learn scipy plotly lifelines
    # Clone the repo (replace with your GitHub URL after pushing)
    # !git clone https://github.com/YOUR_USERNAME/esp-predictive-maintenance.git
    # %cd esp-predictive-maintenance
    from google.colab import drive
    drive.mount('/content/drive')

print('Dependencies ready.')

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# ── Path setup ────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Style
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.grid': True,
    'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11,
})
sns.set_palette('husl')

NORMAL_COLOR  = '#1D9E75'
FAILURE_COLOR = '#E24B4A'
PRED_COLOR    = '#378ADD'

print('Imports OK.')

In [ ]:
# ── Cell 3: Load data ─────────────────────────────────────────────
# Option A: Use synthetic data (always works, no downloads)
USE_SYNTHETIC = True  # Set False if you have the Kaggle dataset

if USE_SYNTHETIC:
    sys.path.insert(0, os.path.abspath('..'))
    from src.data.synthetic_generator import generate_esp_dataset, SYNTHETIC_SENSOR_COLS
    df = generate_esp_dataset(n_wells=30, timesteps_per_well=3000, random_seed=42)
    SENSOR_COLS = SYNTHETIC_SENSOR_COLS
    LABEL_COL = 'machine_status'
    print(f'Synthetic dataset: {len(df):,} rows, {len(SENSOR_COLS)} sensors')
else:
    # Option B: Kaggle Pump Sensor Dataset
    # Download: kaggle datasets download -d nphantawee/pump-sensor-data
    df = pd.read_csv('../data/raw/pump_sensor.csv', parse_dates=['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    SENSOR_COLS = [f'sensor_{i:02d}' for i in range(1, 53)]
    SENSOR_COLS = [c for c in SENSOR_COLS if c in df.columns]
    LABEL_COL = 'machine_status'
    print(f'Pump sensor dataset: {len(df):,} rows, {len(SENSOR_COLS)} sensors')

print(f'Label distribution:')
print(df[LABEL_COL].value_counts())
df.head(3)

In [ ]:
# ── Cell 4: Class imbalance ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
status_counts = df[LABEL_COL].value_counts()
colors = [NORMAL_COLOR if s == 'NORMAL' else FAILURE_COLOR if s == 'BROKEN' else '#EF9F27'
          for s in status_counts.index]
axes[0].pie(status_counts.values, labels=status_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Machine Status Distribution', fontsize=13)

# Timeline
failure_mask = df[LABEL_COL] == 'BROKEN'
axes[1].fill_between(range(len(df)), failure_mask.astype(int),
                     color=FAILURE_COLOR, alpha=0.7, label='BROKEN')
axes[1].fill_between(range(len(df)), (~failure_mask).astype(int),
                     color=NORMAL_COLOR, alpha=0.3, label='NORMAL')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Status')
axes[1].set_title('Failure Events Over Time', fontsize=13)
axes[1].legend()

plt.suptitle('⚠️  Class Imbalance: Failure events are rare by design', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../results/01_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

failure_pct = (df[LABEL_COL] == 'BROKEN').mean() * 100
print(f'Failure rate: {failure_pct:.2f}%  →  Need SMOTE / class weighting!')

In [ ]:
# ── Cell 5: Sensor overview plot ─────────────────────────────────
# Show the first 6 sensors + failure regions
show_sensors = SENSOR_COLS[:6]
n_show = len(show_sensors)

fig, axes = plt.subplots(n_show, 1, figsize=(14, n_show * 2.2), sharex=True)

# Use a single well for clarity
if 'well_id' in df.columns:
    well_df = df[df['well_id'] == df['well_id'].iloc[0]].reset_index(drop=True)
else:
    well_df = df.head(3000)

failure_mask = well_df[LABEL_COL] == 'BROKEN'
x = np.arange(len(well_df))

for i, col in enumerate(show_sensors):
    ax = axes[i]
    ax.plot(x, well_df[col], color=PRED_COLOR, linewidth=0.8, alpha=0.9)
    # Shade failure
    ax.fill_between(x, well_df[col].min(), well_df[col].max(),
                    where=failure_mask, color=FAILURE_COLOR, alpha=0.15,
                    label='Failure' if i == 0 else '')
    ax.set_ylabel(col, fontsize=9, rotation=0, ha='right', labelpad=80)
    ax.tick_params(axis='y', labelsize=8)

axes[-1].set_xlabel('Timestep', fontsize=11)
axes[0].legend(fontsize=9, loc='upper right')
fig.suptitle('ESP Sensor Time-Series with Failure Periods (red shading)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/01_sensor_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 6: Statistical summary ─────────────────────────────────
df['is_failure'] = (df[LABEL_COL] == 'BROKEN').astype(int)

normal_stats = df[df['is_failure'] == 0][SENSOR_COLS].describe().T
failure_stats = df[df['is_failure'] == 1][SENSOR_COLS].describe().T

# Mean shift: which sensors change most during failure?
mean_shift = (failure_stats['mean'] - normal_stats['mean']).abs() / (normal_stats['std'] + 1e-8)
mean_shift = mean_shift.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(mean_shift.index[::-1], mean_shift.values[::-1],
               color=[FAILURE_COLOR if v > 1 else PRED_COLOR for v in mean_shift.values[::-1]],
               alpha=0.85, edgecolor='white')
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.6, label='1σ threshold')
ax.set_xlabel('Normalised mean shift |μ_fail - μ_normal| / σ_normal', fontsize=10)
ax.set_title('Top sensors differentiating FAILURE vs NORMAL', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../results/01_sensor_discrimination.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top discriminating sensors:')
print(mean_shift.to_string())

In [ ]:
# ── Cell 7: Correlation matrix ────────────────────────────────────
# Compute on normal data only (baseline correlations)
normal_df = df[df['is_failure'] == 0][SENSOR_COLS[:12]].dropna()
corr = normal_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap='RdBu_r', vmin=-1, vmax=1,
            center=0, annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.5, square=True)
ax.set_title('Sensor Correlation Matrix (Normal Operating Conditions)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/01_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Domain insight: Strongly correlated sensors → candidates for PCA reduction')

In [ ]:
# ── Cell 8: PCA visualisation ────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sample_df = df.dropna(subset=SENSOR_COLS).sample(min(5000, len(df)), random_state=42)
X = StandardScaler().fit_transform(sample_df[SENSOR_COLS].values)
labels = (sample_df[LABEL_COL] == 'BROKEN').values

pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PC1 vs PC2
for label, color, name in [(False, NORMAL_COLOR, 'Normal'), (True, FAILURE_COLOR, 'Failure')]:
    mask = labels == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, alpha=0.3,
                    s=8, label=name)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=11)
axes[0].set_title('PCA — Sensor Space', fontsize=12)
axes[0].legend(markerscale=3)

# Explained variance
pca_full = PCA(n_components=min(20, len(SENSOR_COLS)), random_state=42)
pca_full.fit(X)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
axes[1].plot(range(1, len(cumvar)+1), cumvar, 'o-', color=PRED_COLOR, linewidth=2)
axes[1].axhline(90, color='gray', linestyle='--', alpha=0.6, label='90% explained')
axes[1].set_xlabel('Number of PCs', fontsize=11)
axes[1].set_ylabel('Cumulative Explained Variance (%)', fontsize=11)
axes[1].set_title('PCA Scree Plot', fontsize=12)
axes[1].legend(fontsize=9)

plt.suptitle('PCA of ESP Sensor Space', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/01_pca.png', dpi=150, bbox_inches='tight')
plt.show()

n_components_90 = np.argmax(cumvar >= 90) + 1
print(f'{n_components_90} PCs explain 90% of variance → useful for dimensionality reduction')

In [ ]:
# ── Cell 9: Failure mode analysis (synthetic only) ────────────────
if 'failure_mode' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    axes = axes.flatten()
    
    mode_cols = {
        'gas_locking':       ('motor_current_A', 'intake_pressure_psi'),
        'abrasive_wear':     ('vibration_x_g',   'intake_pressure_psi'),
        'motor_overheating': ('motor_temperature_C', 'winding_resistance_ohm'),
        'scale_buildup':     ('discharge_pressure_psi', 'flow_rate_bpd'),
    }
    
    for ax, (mode, (col_x, col_y)) in zip(axes, mode_cols.items()):
        mode_df = df[df['failure_mode'] == mode]
        if len(mode_df) == 0:
            continue
        normal_m = mode_df[mode_df['machine_status'] == 'NORMAL']
        broken_m = mode_df[mode_df['machine_status'] == 'BROKEN']
        ax.scatter(normal_m[col_x].sample(min(500, len(normal_m)), random_state=42),
                   normal_m[col_y].sample(min(500, len(normal_m)), random_state=42),
                   c=NORMAL_COLOR, alpha=0.4, s=8, label='Normal')
        ax.scatter(broken_m[col_x].sample(min(500, len(broken_m)), random_state=42),
                   broken_m[col_y].sample(min(500, len(broken_m)), random_state=42),
                   c=FAILURE_COLOR, alpha=0.5, s=10, label='Failure')
        ax.set_xlabel(col_x, fontsize=9)
        ax.set_ylabel(col_y, fontsize=9)
        ax.set_title(mode.replace('_', ' ').title(), fontsize=11)
        ax.legend(fontsize=8, markerscale=2)
    
    plt.suptitle('Sensor Signatures by Failure Mode', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('../results/01_failure_modes.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Cell 10: Key EDA findings summary ────────────────────────────
print('='*65)
print('EDA SUMMARY — Key findings for modelling decisions')
print('='*65)

failure_pct = (df[LABEL_COL] == 'BROKEN').mean() * 100
print(f'\n1. CLASS IMBALANCE: {failure_pct:.2f}% failure events')
print('   → Use SMOTE oversampling and class-weighted loss')
print('   → Prioritise AUC-ROC and F1 over accuracy')

print(f'\n2. SENSOR COUNT: {len(SENSOR_COLS)} sensors')
print(f'   → {n_components_90} PCs explain 90% variance → possible dimensionality reduction')
print('   → Many sensors are correlated (see heatmap)')

print('\n3. TEMPORAL STRUCTURE:')
print('   → Sensor signals show clear pre-failure degradation trends')
print('   → Sliding window (50 timesteps) captures degradation trajectory')

print('\n4. DOMAIN FEATURES TO ADD (notebook 02):')
print('   → Rolling std (vibration instability indicator)')
print('   → Rate of change (current, temperature)')
print('   → Differential pressure (discharge - intake)')
print('   → Pump efficiency proxy (hydraulic / shaft power)')

print('\n5. MODEL CHOICE:')
print('   → Unsupervised: LSTM-AE (no labels needed)')
print('   → Semi-supervised: Transformer-AE')
print('   → RUL: Bi-LSTM regressor')
print('   → Survival: Cox PH + Weibull AFT')
print('='*65)